# 🗺️ Analiza danych przestrzennych z GeoJSON (Apache Sedona + Databricks)

W tym notatniku:
- Wczytujemy dane GeoJSON z `dbfs:/databricks-datasets/nyctaxi/`
- Tworzymy poligon ręcznie
- Obliczamy odległości, powierzchnie, zawieranie i przecięcia geograficzne
- Używamy biblioteki `Apache Sedona`


In [ ]:
# %pip install apache-sedona

from sedona.register import SedonaRegistrator
from sedona.utils import SedonaKryoRegistrator, KryoSerializer
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("GeoJSON Analysis with Sedona") \
    .config("spark.serializer", KryoSerializer.getName) \
    .config("spark.kryo.registrator", SedonaKryoRegistrator.getName) \
    .getOrCreate()

SedonaRegistrator.registerAll(spark)

In [ ]:
geojson_path = "dbfs:/databricks-datasets/nyctaxi/sample_geojson.json"

df_geo = spark.read.format("geojson").load(geojson_path)
df_geo.createOrReplaceTempView("geojson_data")

df_geo.show(5)

In [ ]:
polygon_wkt = "POLYGON((-74.05 40.68, -74.05 40.85, -73.85 40.85, -73.85 40.68, -74.05 40.68))"
polygon_df = spark.sql(f"SELECT ST_GeomFromText('{polygon_wkt}') AS polygon")
polygon_df.createOrReplaceTempView("polygon_data")
polygon_df.show()

In [ ]:
spark.sql("""
SELECT ST_Area(polygon) AS area
FROM polygon_data
""").show()

In [ ]:
spark.sql("""
SELECT ST_Distance(
    ST_Point(-74.0, 40.75), 
    ST_Point(-73.95, 40.78)
) AS distance
""").show()

In [ ]:
spark.sql("""
SELECT g.*, ST_Contains(p.polygon, ST_GeomFromGeoJSON(g.geometry)) AS is_within
FROM geojson_data g
JOIN polygon_data p
""").show()

In [ ]:
spark.sql("""
SELECT g.*, 
       ST_Intersects(p.polygon, ST_GeomFromGeoJSON(g.geometry)) AS intersects,
       ST_Within(ST_GeomFromGeoJSON(g.geometry), p.polygon) AS within
FROM geojson_data g
JOIN polygon_data p
""").show()